In [1]:
import os
import json
import pickle
import tqdm

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv()

In [2]:
MODEL = "google/gemini-2.5-flash"

In [ ]:
df = pd.read_excel("data_clean.xlsx", header=0)

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can extract short and concise tags for student projects title in russian.
Please, for a given student project title return a list of corresponding tags, which depicts the field of science to which the project is relevant.
For example, for a project title "Рекомендательная система для студенческих проектов" you answer might be the following list of strings with underscores.
["machine_learning", "recommender_systems", "software_engineering"]
Also another example, for a project title "Прогнозирование эффектов процентной политики Банка России" you answer might be the following list of strings with underscores.
["economics", "macroeconomics", "monetary_policy", "forecasting", "time_series"]
Return strictly a list of strings in a snake_case in English as an answer.
"""

In [ ]:
messages = []

In [ ]:
for row in df.iterrows():
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[1]["title_rus"]},
        )
    )

In [7]:
tags_set = set()

In [8]:
titles_with_tags_dict = dict()

In [ ]:
client = OpenAI(
    api_key=os.environ.get("KODIK_API_KEY"),
    base_url="https://api.kodikrouter.ru/v1",
)

In [ ]:
for message in tqdm.tqdm(messages):

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=message,
        )

        answer = json.loads(response.choices[0].message.content)

        titles_with_tags_dict[message[1]["content"]] = answer
        tags_set = tags_set.union(answer)

    except Exception as e:

        print(f'{message[1]["content"]} was not processed due to {e}')

In [ ]:
len(tags_set)

In [ ]:
list(titles_with_tags_dict.values())[0]

In [ ]:
list(titles_with_tags_dict.values())[-1]

In [4]:
output_dir_name = MODEL.split("/")[1]

In [6]:
if not os.path.exists(output_dir_name):
    os.mkdir(output_dir_name)

In [9]:
with open(f"{output_dir_name}/titles_with_tags_dict.pkl", "wb") as f:
    pickle.dump(titles_with_tags_dict, f)

In [10]:
with open(f"{output_dir_name}/tags_set.pkl", "wb") as f:
    pickle.dump(tags_set, f)